In [4]:
# Simple NLP Text Classification using NLTK + Naive Bayes
# Dataset: 20 Newsgroups
#
# Main steps:
# 1. Load the data
# 2. Clean/tokenize text using NLTK
# 3. Convert text to TF-IDF features
# 4. Train Naive Bayes
# 5. Evaluate the model
# 6. Predict new text

# Install once if needed:
# pip install nltk scikit-learn

import nltk
import numpy as np





In [5]:

nltk.download("punkt")
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [6]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()


def nltk_tokenizer(text):

    text = text.lower()
    words = word_tokenize(text)
    words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word.isalpha() and word not in stop_words
    ]

    return words


In [7]:
from sklearn.datasets import fetch_20newsgroups

train_data = fetch_20newsgroups(
    subset="train",
    remove=("headers", "footers", "quotes")
)

print("Number of documents:", len(train_data.data))
print("Number of classes:", len(train_data.target_names))

print("\nClasses:")
print(train_data.target_names)

Number of documents: 11314
Number of classes: 20

Classes:
['alt.atheism', 'comp.graphics', 'comp.os.ms-windows.misc', 'comp.sys.ibm.pc.hardware', 'comp.sys.mac.hardware', 'comp.windows.x', 'misc.forsale', 'rec.autos', 'rec.motorcycles', 'rec.sport.baseball', 'rec.sport.hockey', 'sci.crypt', 'sci.electronics', 'sci.med', 'sci.space', 'soc.religion.christian', 'talk.politics.guns', 'talk.politics.mideast', 'talk.politics.misc', 'talk.religion.misc']


In [8]:
from sklearn.model_selection import train_test_split


X_train, X_val, y_train, y_val = train_test_split(
    train_data.data,
    train_data.target,
    test_size=0.20,
    random_state=1
)

print("\nTraining documents:", len(X_train))
print("Validation documents:", len(X_val))



Training documents: 9051
Validation documents: 2263


In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline


model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            tokenizer=nltk_tokenizer,
            token_pattern=None
        )
    ),
    (
        "naive_bayes",
        MultinomialNB(alpha=0.1)
    )
])


In [12]:

model.fit(X_train, y_train)

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Pipeline(steps=[('tfidf',
                 TfidfVectorizer(token_pattern=None,
                                 tokenizer=<function nltk_tokenizer at 0x7d1a45b2bec0>)),
                ('naive_bayes', MultinomialNB(alpha=0.1))])

In [13]:
y_pred = model.predict(X_val)

In [14]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

accuracy = accuracy_score(y_val, y_pred)
f1 = f1_score(y_val, y_pred, average="macro")

print("\nResults")
print("-------")
print("Accuracy:", round(accuracy, 4))
print("Macro F1:", round(f1, 4))

print("\nClassification Report:")
print(
    classification_report(
        y_val,
        y_pred,
        target_names=train_data.target_names
    )
)


Results
-------
Accuracy: 0.7274
Macro F1: 0.7137

Classification Report:
                          precision    recall  f1-score   support

             alt.atheism       0.71      0.45      0.55        97
           comp.graphics       0.74      0.61      0.67       114
 comp.os.ms-windows.misc       0.70      0.70      0.70       112
comp.sys.ibm.pc.hardware       0.63      0.69      0.66       127
   comp.sys.mac.hardware       0.82      0.81      0.82       112
          comp.windows.x       0.72      0.84      0.78       115
            misc.forsale       0.83      0.68      0.75       124
               rec.autos       0.73      0.70      0.72       108
         rec.motorcycles       0.46      0.81      0.59        99
      rec.sport.baseball       0.86      0.82      0.84       113
        rec.sport.hockey       0.90      0.87      0.88       108
               sci.crypt       0.71      0.88      0.79       120
         sci.electronics       0.78      0.63      0.70       119


In [ ]:
def classify_text(text):
    probabilities = model.predict_proba([text])[0]
    best_class = np.argmax(probabilities)
    best_probability = probabilities[best_class]
    print("\nText:", text)
    print("Predicted topic:", train_data.target_names[best_class])
    print("Probability:", round(best_probability, 4))

In [15]:
classify_text("My first homemade PCB. My SMD soldering skills are not great.")
classify_text("New Toyota car launch has been delayed.")
classify_text("Particles from Earth can create water on the lunar surface.")





Text: My first homemade PCB. My SMD soldering skills are not great.
Predicted topic: sci.electronics
Probability: 0.4861

Text: New Toyota car launch has been delayed.
Predicted topic: rec.autos
Probability: 0.6575

Text: Particles from Earth can create water on the lunar surface.
Predicted topic: sci.space
Probability: 0.7943


In [16]:

# ---------------------------------------------------------
# 9. Optional: train on all training data and test
# ---------------------------------------------------------
final_model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            tokenizer=nltk_tokenizer,
            token_pattern=None
        )
    ),
    (
        "naive_bayes",
        MultinomialNB(alpha=0.1)
    )
])

final_model.fit(train_data.data, train_data.target)

test_data = fetch_20newsgroups(
    subset="test",
    remove=("headers", "footers", "quotes")
)

test_pred = final_model.predict(test_data.data)

print("\nFinal Test Results")
print("------------------")
print(
    "Accuracy:",
    round(accuracy_score(test_data.target, test_pred), 4)
)

print(
    "Macro F1:",
    round(f1_score(test_data.target, test_pred, average="macro"), 4)
)


Final Test Results
------------------
Accuracy: 0.6892
Macro F1: 0.6648
